# Build a Meeting-Notes Summarizer

This notebook is a runnable companion to the course's [Build a Meeting-Notes Summarizer](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/meeting-notes-summarizer) project. It mirrors the real pipeline in [`examples/meeting-notes-summarizer/summarize.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/meeting-notes-summarizer/summarize.py): take a plain-text meeting transcript, ask a free-tier LLM for a **structured JSON summary** (decisions, action items, open questions), defensively parse and validate whatever comes back, and render it as readable Markdown.

**How to use this notebook:** run the cells top to bottom. You'll be prompted for a free-tier API key partway through (see the Setup section of the lesson for where to get one — GitHub Models, Gemini, Groq, Mistral, Cerebras, and OpenRouter all work; GitHub Models is the suggested default since it needs no separate signup). Nothing here needs a GPU — this works fine on Colab's free CPU runtime, Kaggle Notebooks, or Binder.

## 1. Install dependencies

In [ ]:
!pip install -q openai>=1.59.0

## 2. Sample transcripts

These are two of the three real sample transcripts shipped with the example (`examples/meeting-notes-summarizer/sample_transcripts/`), embedded directly as strings so this notebook runs with zero file uploads.

In [ ]:
STANDUP_TRANSCRIPT = "Daily Standup -- Payments Team -- Tuesday\n\nMaria: Morning everyone, let's keep this quick. James, how's the API migration going?\nJames: About 70% done. I should finish the auth endpoints by Friday. I'll also write up the migration guide for the team once that's done.\nMaria: Great, thanks. Priya, what about the invoice export bug?\nPriya: Found the root cause yesterday -- it's a timezone issue in how we format the PDF timestamps. Fix is small, I'll have a PR up by end of day.\nMaria: Nice. Can you also add a regression test for it so we don't hit this again?\nPriya: Yep, will do -- I'll add that in the same PR.\nJames: Quick question -- are we still planning to deprecate the v1 endpoints next month?\nMaria: Let's hold off on committing to a date for that. I want to see the v2 adoption numbers first before we lock anything in.\nPriya: Makes sense. Separately, has anyone heard back from the infra team about the staging environment being slow?\nMaria: Not yet, I'll ping them after this and follow up.\nJames: Sounds good. I think that's everything from me.\nMaria: Great, let's also revisit next week whether we need a dedicated on-call rotation for the payments service -- we haven't decided on that yet.\nPriya: Agreed, worth a real conversation, not just a standup mention.\nMaria: Okay, that's a wrap. Thanks everyone.\n"

INCIDENT_REVIEW_TRANSCRIPT = "Incident Review -- Checkout Service Outage (2026-07-18)\n\nNadia: Let's walk through the timeline. The outage started at 14:02 UTC when checkout error rates spiked to 80%.\nTomas: Right, and we didn't get paged until 14:11 because the alert threshold was set too high. That's the first thing we need to fix.\nNadia: Agreed -- let's lower the checkout error-rate alert threshold from 50% to 10%. I'll make that change myself this week.\nTomas: Root cause was the new rate limiter deploy at 13:55 -- it was misconfigured and started rejecting legitimate checkout requests, not just abusive traffic.\nElena: Should we roll back deploys automatically when error rates spike like that, or is that too risky to automate right now?\nNadia: That's a good question, and I don't think we should decide it today -- it needs a broader discussion with the platform team about failure modes. Let's leave it open for now.\nTomas: Fair. In the meantime, I'll add a canary stage to the rate limiter's deploy pipeline so a bad config only hits a small percentage of traffic before a full rollout.\nNadia: Good, let's do that -- Tomas, that's on you, no firm deadline yet but let's aim for before the end of the month.\nElena: What about customer comms? We didn't post a status page update until 14:40, almost 40 minutes after the outage started.\nNadia: Agreed that's too slow. Let's commit to updating the status page within 10 minutes of any confirmed sev-1 going forward -- that's now our standard.\nElena: I can update the incident runbook to reflect that new 10-minute target.\nNadia: Thanks Elena, please do.\nTomas: One open item -- do we actually know the total revenue impact yet? I haven't seen final numbers.\nNadia: Not yet, finance is still calculating that. Let's leave it open until they get back to us.\nNadia: Okay, I think we've covered the key points. Thanks everyone for a thorough review.\n"

print(STANDUP_TRANSCRIPT[:200] + "...")

## 3. Set your free-tier API key

Paste a free-tier API key when prompted — it's read with `getpass` so it never gets echoed to the notebook output or saved in the `.ipynb` file. See the lesson's [Setup section](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/meeting-notes-summarizer#3-get-a-free-llm-api-key) for where to get a key for each of the six supported providers. This notebook defaults to **GitHub Models** (a GitHub personal access token with the `models: read` scope) since it needs no separate signup.

In [ ]:
import getpass
import os

# Change this if you'd rather use a different free-tier provider (see PROVIDERS below).
LLM_PROVIDER = "github"

_key_env_var = {
    "github": "GITHUB_TOKEN",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "cerebras": "CEREBRAS_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
}[LLM_PROVIDER]

os.environ[_key_env_var] = getpass.getpass(f"Enter your {LLM_PROVIDER} API key ({_key_env_var}): ")

## 4. Provider setup

Same six-provider pattern as `summarize.py`: every provider exposes an OpenAI-compatible chat completions API, so one client library (`openai`) works across all of them — just pointed at a different `base_url` and model name.

In [ ]:
from openai import OpenAI


def _github_client():
    return OpenAI(api_key=os.environ["GITHUB_TOKEN"], base_url="https://models.github.ai/inference"), "gpt-4o-mini"


def _gemini_client():
    return OpenAI(
        api_key=os.environ["GOOGLE_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    ), "gemini-2.5-flash"


def _groq_client():
    return OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1"), "llama-3.3-70b-versatile"


def _mistral_client():
    return OpenAI(api_key=os.environ["MISTRAL_API_KEY"], base_url="https://api.mistral.ai/v1"), "mistral-small-latest"


def _cerebras_client():
    return OpenAI(api_key=os.environ["CEREBRAS_API_KEY"], base_url="https://api.cerebras.ai/v1"), "llama-3.3-70b"


def _openrouter_client():
    return OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1"
    ), "meta-llama/llama-3.3-70b-instruct:free"


PROVIDERS = {
    "github": _github_client,
    "gemini": _gemini_client,
    "groq": _groq_client,
    "mistral": _mistral_client,
    "cerebras": _cerebras_client,
    "openrouter": _openrouter_client,
}

client, model = PROVIDERS[LLM_PROVIDER]()
print(f"Using provider={LLM_PROVIDER!r} model={model!r}")

## 5. Design the structured-extraction prompt

The core skill this project teaches: ask the model for **JSON in a specific shape** — a schema you define — instead of a free-form paragraph, so the output is something your own code can reliably parse, store, and act on. `owner` is explicitly allowed to be `null`, with an explicit rule for when to use it, so smaller models don't invent a plausible-sounding name or write the string `"TBD"`.

In [ ]:
SYSTEM_PROMPT = """You are an assistant that extracts structured information \
from meeting transcripts. You always respond with a single JSON object and \
nothing else -- no markdown code fences, no commentary before or after it."""

JSON_SCHEMA_DESCRIPTION = """Respond with a JSON object with EXACTLY these keys:

{
  "decisions": ["short string describing one decision that was made", ...],
  "action_items": [
    {"task": "short string describing the task", "owner": "person's name, or null if not stated"},
    ...
  ],
  "open_questions": ["short string describing one unresolved question", ...]
}

Rules:
- Only include a decision if the transcript shows the group actually agreeing on something -- not just discussing an option.
- Only include an action item if someone (or the group) commits to doing it.
- "owner" must be null (not the string "null", not "TBD") when no specific person is named for that task.
- If a category has nothing to report, use an empty list -- never omit the key.
- Do not invent information that isn't in the transcript."""


def build_prompt(transcript: str) -> list[dict]:
    """Returns the chat messages list ready to send to the LLM."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{JSON_SCHEMA_DESCRIPTION}\n\nTranscript:\n{transcript}"},
    ]

## 6. Call the LLM

In [ ]:
def call_llm(transcript: str) -> str:
    """Sends the structured-extraction prompt and returns the model's raw text reply."""
    response = client.chat.completions.create(
        model=model,
        messages=build_prompt(transcript),
        temperature=0,  # deterministic-as-possible extraction, not creative writing
    )
    return response.choices[0].message.content

## 7. Parse and validate the JSON response

Even a well-designed prompt occasionally gets a response wrapped in a ```` ```json ```` code fence, with a trailing comment, or with a stray comma — and a naive `json.loads()` call crashes on all three. `extract_json` strips the common wrapping; `parse_summary` validates the schema and raises a clear error (with the raw text attached) if it still doesn't match, instead of silently returning an empty summary.

In [ ]:
import json
import re

REQUIRED_KEYS = {"decisions", "action_items", "open_questions"}


def extract_json(raw_text: str) -> str:
    """Strips common wrapping the model adds around JSON despite being told not to."""
    text = raw_text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    if fenced:
        return fenced.group(1).strip()
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start : end + 1]
    return text


def parse_summary(raw_text: str) -> dict:
    """Parses and validates the model's response, raising a clear error if it
    doesn't match the schema after the best-effort cleanup in extract_json()."""
    cleaned = extract_json(raw_text)
    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError as error:
        raise ValueError(
            f"Model response wasn't valid JSON even after cleanup: {error}\nRaw response was:\n{raw_text}"
        ) from error

    if not isinstance(data, dict) or not REQUIRED_KEYS.issubset(data.keys()):
        raise ValueError(f"Response is missing required keys {REQUIRED_KEYS}. Got: {data!r}")

    for key in ("decisions", "action_items", "open_questions"):
        if not isinstance(data[key], list):
            data[key] = [data[key]]

    return data

## 8. Format the result as readable Markdown

In [ ]:
def format_markdown(summary: dict, source: str) -> str:
    """Renders a parsed summary dict as short, skimmable Markdown."""
    lines = [f"# Meeting Summary — {source}", ""]

    lines.append("## Decisions")
    if summary["decisions"]:
        lines += [f"- {d}" for d in summary["decisions"]]
    else:
        lines.append("_No decisions recorded._")
    lines.append("")

    lines.append("## Action Items")
    if summary["action_items"]:
        for item in summary["action_items"]:
            owner = item.get("owner") or "unassigned"
            lines.append(f"- [ ] {item['task']} — **{owner}**")
    else:
        lines.append("_No action items recorded._")
    lines.append("")

    lines.append("## Open Questions")
    if summary["open_questions"]:
        lines += [f"- {q}" for q in summary["open_questions"]]
    else:
        lines.append("_No open questions recorded._")

    return "\n".join(lines)

## 9. Run it end to end

Runs the full pipeline on the real `standup.txt` sample transcript embedded above: call the LLM, parse and validate the JSON, then print both the raw parsed `dict` and the formatted Markdown.

In [ ]:
raw_response = call_llm(STANDUP_TRANSCRIPT)
summary = parse_summary(raw_response)

print("=== Parsed JSON ===")
print(json.dumps(summary, indent=2))

print("\n=== Formatted Markdown ===")
print(format_markdown(summary, source="standup.txt"))

## 10. Try it on a second transcript

The incident-review transcript stresses the schema differently — it tends to produce far more open questions than action items.

In [ ]:
raw_response_2 = call_llm(INCIDENT_REVIEW_TRANSCRIPT)
summary_2 = parse_summary(raw_response_2)

print("=== Parsed JSON ===")
print(json.dumps(summary_2, indent=2))

print("\n=== Formatted Markdown ===")
print(format_markdown(summary_2, source="incident_review.txt"))

## Where to go from here

- Run this on your own transcript: replace `STANDUP_TRANSCRIPT` with any speaker-labeled plain-text meeting transcript.
- Run the full local version (all six providers, writes `.json`/`.md` files) from [`examples/meeting-notes-summarizer/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/meeting-notes-summarizer).
- Read the full lesson at [Build a Meeting-Notes Summarizer](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/meeting-notes-summarizer) for the step-by-step walkthrough this notebook mirrors, including common pitfalls and where to go next (adding a `sentiment` field, validating with `pydantic`, wiring this into the [AI Agent project](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/ai-agent) as a tool).